## 🔧 设置

我们导入所需的库：
- 用于数据库的 SQLite
- OpenAI（兼容OpenRouter）
- UI 渐变
- API 密钥的 dotenv

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

MODEL = "openai/gpt-4o-mini"

## 🗄️ 数据库设置

我们创建一个 SQLite 数据库来存储航班价格。
如果表已经存在，则不会重新创建。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
conn = sqlite3.connect("flights.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS flights (
    city TEXT PRIMARY KEY,
    price INTEGER
)
""")

conn.commit()

## 🛠️工具功能

这些是 LLM 可以调用的函数：
- 获取价格（多城市）
- 添加价格（学习能力）

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def get_price(cities):
    results = {}
    for city in cities:
        cursor.execute("SELECT price FROM flights WHERE city = ?", (city,))
        result = cursor.fetchone()

        if result:
            results[city] = result[0]
        else:
            results[city] = "NOT_FOUND"   # 🔥 KEY FIX

    return results


def add_price(city, price):
    if price <= 0:
        return "Invalid price"

    cursor.execute(
        "INSERT OR REPLACE INTO flights (city, price) VALUES (?, ?)",
        (city, price)
    )
    conn.commit()
    return f"Saved {city} → ${price}"

## 🌱 种子初始数据

我们插入一些默认城市和价格。
如果它们已经存在，它们将被忽略。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def seed_data():
    default_data = [
        ("London", 500),
        ("Paris", 450),
        ("New York", 700),
        ("Tokyo", 800),
        ("Dubai", 300)
    ]

    for city, price in default_data:
        cursor.execute(
            "INSERT OR IGNORE INTO flights (city, price) VALUES (?, ?)",
            (city, price)
        )

    conn.commit()
    print("✅ Seed data inserted (if not already present)")


seed_data()

## 🔧 工具定义

我们以法学硕士可以理解的格式定义工具。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_price",
            "description": "Get flight prices for cities",
            "parameters": {
                "type": "object",
                "properties": {
                    "cities": {
                        "type": "array",
                        "items": {"type": "string"}
                    }
                },
                "required": ["cities"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_price",
            "description": "Add or update flight price",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "price": {"type": "integer"}
                },
                "required": ["city", "price"]
            }
        }
    }
]

## 🧠 系统提示

定义助手的行为。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
SYSTEM_PROMPT = """
You are a smart airline assistant.

Rules:
1. If user mentions a single city → assume they want price for that city.
2. Always call get_price with that city.
3. If NOT_FOUND → ask user for price.
4. If user provides price → call add_price.
5. Always respond in markdown.
"""

## 🔁 代理循环

手柄：
- 工具调用
- 多步推理
- 迭代响应

In [ ]:
def chat_with_agent(user_input, history=[]):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    messages.extend(history)
    messages.append({"role": "user", "content": user_input})

    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        msg = response.choices[0].message

        # ✅ 仅当需要工具时
        if msg.tool_calls:

            messages.append(msg)  # 🔥 IMPORTANT (assistant message with tool_calls)

            for tool_call in msg.tool_calls:
                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                if name == "get_price":
                    result = get_price(**args)

                elif name == "add_price":
                    result = add_price(**args)

                # ✅ 工具响应必须遵循 tool_call
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

            continue  # loop again

        else:
            messages.append(msg)
            return msg.content, messages

## 🌐 广播界面

创建用于与代理交互的聊天 UI。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
chat_history = []

def respond(message, history):
    try:
        response, updated_messages = chat_with_agent(message, history)
        return response
    except Exception as e:
        return f"⚠️ Error occurred: {str(e)}"

demo = gr.ChatInterface(
    fn=respond,
    title="✈️ Airline AI Assistant",
    description="Ask for flight prices or teach new ones!"
)

demo.launch()

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
cursor.execute("SELECT * FROM flights")
print(cursor.fetchall())